## CUDA GEMM Benchmark (Colab)

This notebook assumes you copied the repo's `cuda-benchmark/` folder to the root of your Google Drive at:

`MyDrive/cuda-benchmark/`

It will:
- Mount Google Drive
- Compile `cublas_benchmark.cu` with `nvcc`
- Run the PyTorch benchmark
- Run the cuBLAS benchmark
- Run the plotting script (writes plots to `plots/`)


In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# (Optional) sanity checks
!nvidia-smi
!nvcc --version


Fri Mar 20 03:44:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# Go to cuda-benchmark folder in Drive
import os
bench_dir = '/content/drive/MyDrive/cuda-benchmark'
assert os.path.isdir(bench_dir), f"Missing {bench_dir}. Copy the repo's cuda-benchmark folder there."
%cd {bench_dir}
!ls -la


/content/drive/MyDrive/cuda-benchmark
total 31
-rw------- 1 root root 15590 Mar 20 03:38 cublas_benchmark.cu
-rw------- 1 root root  2587 Mar 20 03:44 cuda_benchmark_colab.ipynb
-rw------- 1 root root  6674 Mar 20 02:50 plot_benchmarks.py
-rw------- 1 root root  4937 Mar 20 02:40 pytorch_benchmark.py


In [4]:
# Ensure plotting deps exist (PyTorch is typically preinstalled on Colab GPUs)
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'available', torch.cuda.is_available())"
!pip -q install matplotlib


torch 2.10.0+cu128 cuda 12.8 available True


In [5]:
# Compile cuBLAS benchmark
# Note: Colab has CUDA + cuBLAS available on the system image
!nvcc -O3 --std=c++17 cublas_benchmark.cu -o cublas_benchmark -lcublas
!ls -la


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
total 1041
-rwx------ 1 root root 1034656 Mar 20 03:44 cublas_benchmark
-rw------- 1 root root   15590 Mar 20 03:38 cublas_benchmark.cu
-rw------- 1 root root    2587 Mar 20 03:44 cuda_benchmark_colab.ipynb
-rw------- 1 root root    6674 Mar 20 02:50 plot_benchmarks.py
-rw------- 1 root root    4937 Mar 20 02:40 pytorch_benchmark.py


In [6]:
# Run PyTorch benchmark using script defaults
!python pytorch_benchmark.py


Wrote 18 entries to results/pytorch_benchmark.json


In [7]:
# Run cuBLAS benchmark using binary defaults
!./cublas_benchmark


mode=cuda_core_fp32 hidden=64 avg_ms=0.052 TFLOPs=5.12
mode=cuda_core_fp32 hidden=128 avg_ms=0.085 TFLOPs=6.35
mode=cuda_core_fp32 hidden=256 avg_ms=0.134 TFLOPs=8.04
mode=cuda_core_fp32 hidden=512 avg_ms=0.213 TFLOPs=10.10
mode=cuda_core_fp32 hidden=1024 avg_ms=0.338 TFLOPs=12.72
mode=cuda_core_fp32 hidden=2048 avg_ms=0.509 TFLOPs=16.89
mode=cuda_core_fp32 hidden=4096 avg_ms=0.968 TFLOPs=17.76
mode=cuda_core_fp32 hidden=8192 avg_ms=2.025 TFLOPs=16.97
mode=cuda_core_fp32 hidden=16384 avg_ms=3.994 TFLOPs=17.21
mode=tensor_core_tf32 hidden=64 avg_ms=0.035 TFLOPs=7.74
mode=tensor_core_tf32 hidden=128 avg_ms=0.038 TFLOPs=14.09
mode=tensor_core_tf32 hidden=256 avg_ms=0.045 TFLOPs=23.99
mode=tensor_core_tf32 hidden=512 avg_ms=0.077 TFLOPs=27.89
mode=tensor_core_tf32 hidden=1024 avg_ms=0.094 TFLOPs=45.81
mode=tensor_core_tf32 hidden=2048 avg_ms=0.136 TFLOPs=63.30
mode=tensor_core_tf32 hidden=4096 avg_ms=0.199 TFLOPs=86.21
mode=tensor_core_tf32 hidden=8192 avg_ms=0.342 TFLOPs=100.43
mode=tenso

In [8]:
# Plot comparison curves using script defaults
!python plot_benchmarks.py
!ls -la plots


Saved plots to plots
total 325
-rw------- 1 root root 68653 Mar 20 03:45 combined_latency_vs_hidden.png
-rw------- 1 root root 69969 Mar 20 03:45 combined_tflops_vs_hidden.png
-rw------- 1 root root 51928 Mar 20 03:45 cublas_latency_vs_hidden.png
-rw------- 1 root root 45651 Mar 20 03:45 cublas_tflops_vs_hidden.png
-rw------- 1 root root 47042 Mar 20 03:45 pytorch_latency_vs_hidden.png
-rw------- 1 root root 47320 Mar 20 03:45 pytorch_tflops_vs_hidden.png
